In [ ]:
import os
import datetime
import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# Mounting Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/My Drive/TX_DATA

/content/drive/My Drive/TX_DATA


In [ ]:
# Read in the merged station datasets
dfs = {}
for index in range(0, 6):
  df = pd.read_csv('merged_' + str(index + 1) + '.dat', sep=",", parse_dates=["Date"], index_col="Date")
  dfs['Station' + str(index + 1)] = df


#df4_swc_50 = dfs['Station4'].pop('SWC_50')
#df4_t_50 = dfs['Station4'].pop('T_50')
#df5_swc_20 = dfs['Station5'].pop('SWC_20')

In [ ]:
#Add estimated values for large null fields -- regular clean data is made by skipping until
#Dropping the Flag feature and adding in commenting pop statements aboce
def target_data_split(frame, targ_field, feat_fields):
  df = frame.copy()
  df = df.dropna(subset=[targ_field])
  df = df.dropna(subset=feat_fields)
  target = df[targ_field].values
  x_data = df[feat_fields].values
  return (target, x_data)

In [ ]:
from sklearn.linear_model import LinearRegression

def lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key, to_skip = None):
  LinModel = LinearRegression(fit_intercept=True)
  #fit model
  for key in dfs.keys():
    if (key != targ_frame_key) and (key != to_skip):
      target, x_data = target_data_split(dfs[key],targ_field, feat_fields)
      LinModel.fit(x_data, target)

  #predict
  index = dfs[targ_frame_key][dfs[targ_frame_key][targ_field].isnull()].index
  x_data = dfs[targ_frame_key][feat_fields].loc[index]
  x_data.dropna(inplace = True)
  y_pred = LinModel.predict(x_data.values)
  return pd.Series(data = y_pred, index = x_data.index)

In [ ]:
#Station 4 SWC 50 Pred
targ_field = 'SWC_50'
feat_fields = ['SWC_5','SWC_10','SWC_20']
targ_frame_key = 'Station4'

swc_50_pred = lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key)

print(swc_50_pred)

Date
2015-01-01 00:00:00    0.121284
2015-01-01 01:00:00    0.121284
2015-01-01 02:00:00    0.121359
2015-01-01 03:00:00    0.121333
2015-01-01 04:00:00    0.121333
                         ...   
2021-08-31 20:00:00    0.093203
2021-08-31 21:00:00    0.093828
2021-08-31 22:00:00    0.093778
2021-08-31 23:00:00    0.093854
2021-09-01 00:00:00    0.093854
Length: 58439, dtype: float64


In [ ]:
targ_field = 'T_50'
feat_fields = ['T_5','T_10','T_20']
targ_frame_key = 'Station4'

t_50_pred = lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key)

print(t_50_pred)

Date
2015-01-01 00:00:00    14.603089
2015-01-01 01:00:00    14.558889
2015-01-01 02:00:00    14.551228
2015-01-01 03:00:00    14.479485
2015-01-01 04:00:00    14.434686
                         ...    
2021-08-31 20:00:00    23.488712
2021-08-31 21:00:00    24.056903
2021-08-31 22:00:00    24.786848
2021-08-31 23:00:00    25.639287
2021-09-01 00:00:00    26.410666
Length: 58440, dtype: float64


In [ ]:
#Station 4 SWC 50 Pred
targ_field = 'SWC_20'
feat_fields = ['SWC_5','SWC_10','SWC_50']
targ_frame_key = 'Station5'

swc_20_pred = lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key, to_skip = 'Station4')

print(swc_20_pred)

Date
2015-01-01 00:00:00    0.344028
2015-01-01 01:00:00    0.344028
2015-01-01 02:00:00    0.344028
2015-01-01 03:00:00    0.342832
2015-01-01 04:00:00    0.342832
                         ...   
2016-02-20 08:00:00    0.363784
2016-02-20 09:00:00    0.363784
2016-02-20 10:00:00    0.363784
2016-02-20 11:00:00    0.364540
2016-02-20 12:00:00    0.364540
Length: 9973, dtype: float64


In [ ]:
# add predictions back to frame


dfs['Station4']['SWC_50'] = swc_50_pred.combine(dfs['Station4']['SWC_50'], max, fill_value=0)
dfs['Station4']['T_50'] = t_50_pred.combine(dfs['Station4']['T_50'], max, fill_value=0)
dfs['Station5']['SWC_20'] = swc_20_pred.combine(dfs['Station5']['SWC_20'], max, fill_value=0)


In [ ]:
dfs['Station4']['SWC_50']

Date
2015-01-01 00:00:00    0.121284
2015-01-01 01:00:00    0.121284
2015-01-01 02:00:00    0.121359
2015-01-01 03:00:00    0.121333
2015-01-01 04:00:00    0.121333
                         ...   
2021-08-31 20:00:00    0.093203
2021-08-31 21:00:00    0.093828
2021-08-31 22:00:00    0.093778
2021-08-31 23:00:00    0.093854
2021-09-01 00:00:00    0.093854
Name: SWC_50, Length: 58441, dtype: float64

In [ ]:
dfs['Station4']['T_50']

Date
2015-01-01 00:00:00    14.603089
2015-01-01 01:00:00    14.558889
2015-01-01 02:00:00    14.551228
2015-01-01 03:00:00    14.479485
2015-01-01 04:00:00    14.434686
                         ...    
2021-08-31 20:00:00    23.488712
2021-08-31 21:00:00    24.056903
2021-08-31 22:00:00    24.786848
2021-08-31 23:00:00    25.639287
2021-09-01 00:00:00    26.410666
Name: T_50, Length: 58441, dtype: float64

In [ ]:
dfs['Station5']['SWC_20']

Date
2015-01-01 00:00:00    0.344028
2015-01-01 01:00:00    0.344028
2015-01-01 02:00:00    0.344028
2015-01-01 03:00:00    0.342832
2015-01-01 04:00:00    0.342832
                         ...   
2021-08-31 20:00:00    0.192000
2021-08-31 21:00:00    0.192000
2021-08-31 22:00:00    0.192000
2021-08-31 23:00:00    0.192000
2021-09-01 00:00:00    0.192000
Name: SWC_20, Length: 58441, dtype: float64

In [ ]:
# Drop flag feature and null values. Average the two ppt values

for station, df in dfs.items():
  df_new = df.copy()
  df_new.drop('Flag', axis = 1, inplace = True)
  ppt_x = df_new.pop('Ppt_x')
  ppt_y = df_new.pop('Ppt_y')
  df_new['Ppt'] = (ppt_x + ppt_y)/2
  df_new.dropna(inplace=True)
  dfs[station] = df_new

In [ ]:
for key in dfs.keys():
  print(dfs[key].shape)

(57586, 14)
(56364, 14)
(58365, 14)
(58307, 14)
(57734, 14)
(58340, 14)


In [ ]:
# Vectorize wind

for station, df in dfs.items():
  wv = df.pop('Windspeed')
  wd_rad = df.pop('Winddirection')*np.pi / 180
  df['Wx'] = wv*np.cos(wd_rad)
  df['Wy'] = wv*np.sin(wd_rad)
  dfs[station] = df

In [ ]:
day = 24*60*60
year = (365.2425)*day

for station, df in dfs.items() :
  timestamp_s = (df.index).map(pd.Timestamp.timestamp)
  df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
  df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
  df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
  df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))
  dfs[station] = df

In [ ]:
#Normalize

for key in dfs.keys():
  dfs[key] = (dfs[key] - dfs[key].min())/(dfs[key].max()-dfs[key].min())

In [ ]:
# Add geographical position data

position_dict = {"Station1": [30.3989,-98.6105],
                 "Station2": [30.4193,-98.8046],
                 "Station3": [30.4421,-98.8427],
                 "Station4": [30.4600, -98.9407],
                 "Station5": [30.2454,-98.7059],
                 "Station6": [30.2758,-98.7242]
                 }

for key in dfs.keys():
  dfs[key]["Latitude"] = position_dict[key][0]
  dfs[key]["Longitude"] = position_dict[key][1]

In [ ]:
# Normalize position (Normalization based on lat range from -90 to 90 and long range -180 to 180)

for station, df in dfs.items():
  df["Latitude"] = (df["Latitude"] + 90)/(180)
  df["Longitude"] = (df["Longitude"] + 180)/(360)

In [ ]:
for key in dfs.keys():
  print(dfs[key].shape)


(57586, 20)
(56364, 20)
(58365, 20)
(58307, 20)
(57734, 20)
(58340, 20)


In [ ]:
for key in dfs.keys():
  dfs[key].to_csv(key + '-Simulated-cleaned-merged-data.csv')

